# Full anonymization pipeline demo

Separation -> blurring -> remixing using `source_separation` + `voice_blurring`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import librosa
import numpy as np
from IPython.display import Audio, display

for _root in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (_root / "anonymization_pipeline").is_dir():
        sys.path.insert(0, str(_root))
        REPO_ROOT = _root
        break
else:
    raise RuntimeError("Repo root not found")

from anonymization_pipeline import anonymize_audio
from source_separation import load_unet_checkpoint


In [4]:
AUDIO_PATH = Path(os.environ.get("PIPE_AUDIO_PATH", "mix_demo.wav"))
CKPT_PATH = Path(os.environ.get("PIPE_CKPT_PATH", str(REPO_ROOT / "checkpoints/unet_run1/unet_voice_sep.pt")))
BLUR_MODE = os.environ.get("PIPE_BLUR_MODE", "mfcc")  # cascade | low_pass | mfcc

if not AUDIO_PATH.is_file():
    raise FileNotFoundError("Set PIPE_AUDIO_PATH or edit AUDIO_PATH")
if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

y, sr = librosa.load(str(AUDIO_PATH), sr=None, mono=True)
y = np.asarray(y, dtype=np.float32)
model, config, device = load_unet_checkpoint(CKPT_PATH, device="auto")

res = anonymize_audio(y, sr, model=model, config=config, device=device, blur_mode=BLUR_MODE)
print(f"Device: {device} | blur_mode={res.blur_mode} | sr={res.sr}")
print(f"lens: mix={len(y)} voice={len(res.voice_est)} blurred={len(res.blurred_voice)} remix={len(res.anonymized_mix)}")


Device: cuda | blur_mode=mfcc | sr=16000
lens: mix=160000 voice=160000 blurred=160000 remix=160000


In [5]:
print("Input mix")
display(Audio(y, rate=sr))
print("Separated voice")
display(Audio(res.voice_est, rate=res.sr))
print("Blurred voice")
display(Audio(res.blurred_voice, rate=res.sr))
print("Anonymized remix")
display(Audio(res.anonymized_mix, rate=res.sr))


Input mix


Separated voice


Blurred voice


Anonymized remix
